In [2]:
import torch
import torch.nn as nn
import torchvision.models as models

# 1. 뼈대 모델 로드
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print(f'model: {model}')

# [추적 1] 순전파 전 뼈대 레이어 첫 층의 가중치 복사
initial_conv1_weight = model.conv1.weight.clone().detach()

# 2. 뼈대 레이어 전체 동결
for param in model.parameters():
    param.requires_grad = False

# 3. 새로운 분류기 레이어로 교체
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

# [추적 2] 순전파 전 새로 생성된 분류기 레이어의 가중치 복사
initial_fc_weight = model.fc.weight.clone().detach()

# 4. 가상 데이터 및 손실함수/옵티마이저 설정
inputs = torch.randn(1, 3, 224, 224)
labels = torch.tensor([1])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.fc.parameters(), lr=0.1)

# 5. 순전파, 역전파, 파라미터 업데이트
optimizer.zero_grad()
outputs = model(inputs)
loss = criterion(outputs, labels)
loss.backward()
optimizer.step()

# 6. 최종 결과 검증 출력
print("=====================================================================")
print("  전이 학습 전/후 레이어별 파라미터(Weight) 변화 검증")
print("=====================================================================")

print("\n■ 1. 동결된 뼈대 레이어 (ResNet18 conv1 층)")
print(f" - 그래디언트(grad) 상태 : {model.conv1.weight.grad}")
is_conv1_same = torch.equal(initial_conv1_weight, model.conv1.weight)
print(f" - 학습 전/후 가중치 동일 여부 : 【 {is_conv1_same} 】 (가중치 고정됨)")

print("\n■ 2. 새로 교체한 분류기 레이어 (fc 층)")
print(f" - 그래디언트(grad) 상태 : {model.fc.weight.grad is not None} (계산됨)")
is_fc_same = torch.equal(initial_fc_weight, model.fc.weight)
print(f" - 학습 전/후 가중치 동일 여부 : 【 {is_fc_same} 】 (가중치 업데이트됨)")
print("=====================================================================")

model: ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_runn